In [ ]:
%reload_ext autoreload
# %load_ext autoreload
%autoreload 2
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import itertools
from g_anomaly import PA_with_anomaly_numba
from g_standard import standard_PA_numba
from support_tools import (
    edge_degree_counting,
    edge_degree_counting_many,
    expected_degree_PA_normal_all_vertices,
    graph_series_connected_vertex_degree,
    get_top_vertices,
    prepare_screening,
    get_vertex_ranks
)
from estimation_pa_anomaly_joint import iteration_estimation_update
from estimation_pa_anomaly_fixed import single_estimation_beta
from estimation_pa_normal import optimize_parameters_normal
from pa_loglikelihood import neg_log_likelihood_normal, neg_log_likelihood_anomaly

import math, time, os, csv, copy
import pylab as plt
import matplotlib.pyplot as plt
import seaborn as sns

from collections import Counter, OrderedDict, defaultdict
import os
from concurrent.futures import ThreadPoolExecutor, as_completed


In [ ]:
def detetable_region(label, gammas, t, type_2_error, m, delta):
    
    betas = []
    
    for gamma in gammas:

        if label == "early":
            tau = math.pow(t,gamma)
        elif label =="midway":    
            tau = gamma * t
        else:   # label = "late"
            tau = t - math.pow(t,gamma)
        
        K = math.exp(math.log(type_2_error) / (t - tau))
        beta = (2 * m + delta) * (1 - K) / K
        betas.append(beta)
    betas = np.array(betas)
    return betas

In [ ]:
###########    find the undetectable region   #########################
label_test = "midway"
error_test = 0.2
network_size_1 = 100000
m_tmp = 3
delta_tmp = 0.1
alpha_test = np.linspace(1e-2, 0.999, 1000)
beta_1 = detetable_region(label_test, alpha_test,network_size_1,error_test, m_tmp, delta_tmp)

# x :alpha
x = alpha_test

# example points
alpha_point_1 = 0.5
beta_point_1 = 0.005
x_point_1 = alpha_point_1

alpha_point_2 = 0.5
beta_point_2 = 0.001
x_point_2 = alpha_point_2

plt.rcParams.update({
    "font.size": 18,
    "axes.titlesize": 18,
    "axes.labelsize": 18,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18
})

plt.figure(figsize=(10, 6), dpi=100)
plt.loglog(x, beta_1, color = "red", label=r'Boundary')

plt.fill_between(x, beta_1, 1e-5, where=(x > 0), color="lightcoral", alpha=0.4, label="Undetectable area")

# mark specific points
plt.scatter(x_point_1, beta_point_1, color="blue", s=20, marker="o")
plt.text(x_point_1 *1.05 , beta_point_1* 0.8, fr"$({alpha_point_1}, {beta_point_1})$", color="blue",ha="right", va="top")

plt.scatter(x_point_2, beta_point_2, color="blue", s=20, marker="o")
plt.text(
    x_point_2*1.05, beta_point_2 * 0.8, 
    fr"$({alpha_point_2}, {beta_point_2})$",
    color="blue",
    ha="right", va="top"
)


plt.xlabel(r'$\gamma$')
plt.ylabel(r'$\beta$')
plt.title(fr"Undetectable region ({label_test} anomaly)")
plt.legend(loc="upper left")
plt.ylim(top=0.5)
plt.tight_layout()
#plt.savefig(f"Undetectable_area_{label_test}_{network_size_1}.pdf", bbox_inches="tight")
plt.show()

In [ ]:
################ Generating graphs to get the rank  #####################
network_size_tmp = 100000
m_tmp = 3
delta_true = 0.1
#vertex_tau_list = [6,15,39,100,251,1000,3000,5000,7000,9000,9300,9500,9700]  # for network size = 10000
vertex_tau_list = [10,31,100,316,999,10000,30000,50000,70000,90000,93000,95000,97000,99000]
beta_list = [0.008,0.01,0.05]

exp_degree_tmp = expected_degree_PA_normal_all_vertices(network_size_tmp, delta_true, m_tmp)
Top_degree_rank = 5
Top_ratio_rank = 5

final_data = []


total_combinations = len(beta_list) * len(vertex_tau_list)

for b in beta_list:
    for tau_true in vertex_tau_list:
        count_d = 0
        count_r = 0
        for rep_1 in range(200):
            seed_1 = int(b * network_size_tmp + tau_true * 100 + rep_1)
            edges_1 = PA_with_anomaly_numba(network_size_tmp, m_tmp, b, tau_true, delta_true, seed_1)
            
            vertex_list_d = get_top_vertices(edges_1, exp_degree_tmp, Top_degree_rank, 0, 0)
            vertex_list_r = get_top_vertices(edges_1, exp_degree_tmp, 0, Top_ratio_rank, 0)
            
            tau_index = tau_true - 1
            if tau_index in vertex_list_d:
                count_d += 1
            if tau_index in vertex_list_r:
                count_r += 1
        
        final_data.append({
            'Beta': b,
            'Tau_True': tau_true,
            'Degree_Capture_Rate': count_d / 200,
            'Ratio_Capture_Rate': count_r / 200
        })

df = pd.DataFrame(final_data)
#df.to_csv(f'screening_experiment_{network_size_tmp}.csv', index=False, encoding='utf-8-sig')

In [ ]:
import matplotlib.pyplot as plt

beta_list_1 = [0.008,0.01,0.05]   

plt.figure(figsize=(12, 7))

my_strong_colors = ['#007AFF', '#FF3B30', '#660099']  

for i, b in enumerate(beta_list_1):
    sub_df = df[df['Beta'] == b].sort_values('Tau_True')
    
    current_color = my_strong_colors[i % len(my_strong_colors)]

    # --- Degree  ---
    plt.plot(sub_df['Tau_True'], sub_df['Degree_Capture_Rate'], 
             label=f'Beta={b} (Degree)', 
             linestyle='-', marker='o', color=current_color, 
             linewidth=2, markersize=5)
    
    # --- Ratio  ---
    plt.plot(sub_df['Tau_True'], sub_df['Ratio_Capture_Rate'], 
             label=f'Beta={b} (Ratio)', 
             linestyle='--', marker='s', color=current_color, 
             linewidth=3, markersize=8)

plt.xscale('log')

plt.ylim(-0.05, 1.05)

plt.legend(loc='lower center', 
           bbox_to_anchor=(0.5, 1.02), 
           ncol=3, 
           fontsize=16,          
           frameon=True, 
           edgecolor='black', 
           handlelength=3, 
           markerscale=1.2)

plt.xticks(fontsize=18)
plt.yticks(fontsize=18)
plt.xlabel(r'$\tau$', fontsize=18)
plt.ylabel('Fraction', fontsize=18)

plt.tight_layout(rect=[0, 0, 1, 0.95]) 

plt.show()

In [ ]:
############ contour plot for log-likelihood function  ####################


delta_true = 0.1
beta_true  = 1.0
vertex_tau = 20000
network_size_tmp = 100000
m_tmp = 5
seed_1 = vertex_tau
edge_list_tmp = PA_with_anomaly_numba(network_size_tmp, m_tmp, beta_true, vertex_tau, delta_true, seed_1)
k_series = graph_series_connected_vertex_degree(edge_list_tmp, receiver_index=1)
k_series = np.asarray(k_series, dtype=np.float64)

degree_series_tau, edge_adjacency_tau = edge_degree_counting(vertex_tau -1 , edge_list_tmp)
degree_series_tau = np.asarray(degree_series_tau, dtype=np.float64)
edge_adjacency_tau = np.asarray(edge_adjacency_tau, dtype=np.float64)

# --- Define parameter grids ---
beta_grid = np.linspace(0.001,2.0, 100)
delta_grid = np.linspace(-1.0, 4.0, 100)

# --- Create meshgrid for plotting ---
B, D = np.meshgrid(beta_grid, delta_grid)
LL = np.zeros_like(B)

# --- Compute log-likelihood for each (β, δ) ---
for i in range(B.shape[0]):
    for j in range(B.shape[1]):
        LL[i, j] = neg_log_likelihood_anomaly(
            B[i, j],
            D[i, j],
            network_size_tmp,
            vertex_tau,
            k_series,
            degree_series_tau,
            edge_adjacency_tau,
            m_tmp
        )

#### LL is already negative

# ===============================================================
#  2D Contour Plot
# ===============================================================
plt.figure(figsize=(10, 6))
contour = plt.contourf(B, D, -LL, levels=60, cmap='viridis')
plt.colorbar(contour, label='Log-Likelihood Value')
plt.xlabel(r"$\beta$", fontsize=12)
plt.ylabel(r"$\delta$", fontsize=12)
plt.title(r"Log-Likelihood Contour for $(\beta, \delta)$", fontsize=14)

plt.scatter(beta_true, delta_true, color='red', s=60, marker='x',
            label=rf"True parameter: $\beta={beta_true:.2f}$, $\delta={delta_true:.2f}$")
plt.legend()
plt.tight_layout()
#plt.savefig("Contour of log-likelihood function.png") 
plt.show()